Complete FastAPI Deployment with Pipeline

Production-ready API with health checks, versioning, and error handling



In [1]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import os


X, y = make_classification(n_samples = 3000, n_features = 10, n_informative=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100))
])

pipeline.fit(X_train, y_train)

# save everything not just the model
os.makedirs('models/v1', exist_ok=True)
joblib.dump(pipeline, 'models/v1/pipeline.pkl')
print(f'Pipeline saved: Accuracy: {pipeline.score(X_test,y_test):.3f}')

# from sklearn.metrics import classification_report, confusion_matrix

# y_pred = pipeline.predict(X_test)

# print("\n" + "=" * 60)
# print("THE PROOF (Confusion Matrix)")
# print("=" * 60)
# print(confusion_matrix(y_test, y_pred))

# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))


# Step 2 - FAST API SERVER

import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="ML PREDICTION API",version='1.0')

# Load at startup, not per request
MODEL_VERSION = os.getenv("MODEL_VERSION", "v1")
pipeline = joblib.load(f"models/{MODEL_VERSION}/pipeline.pkl")

class PredictRequest(BaseModel):
  features: list[float]

class PredictResponse(BaseModel):
  model_config = {"protected_namespaces":()}
  prediction: int
  probability : float
  model_verison : str

@app.post("/predict", response_model = PredictResponse)
def predict(request: PredictRequest):
  try:
    X = np.array(request.features).reshape(1,-1)
    pred = int(pipeline.predict(X[0]))
    prob = float(pipeline.predict_proba(X)[0,1])
    return PredictResponse(
        prediction=pred, probability=prob,
        model_version=MODEL_VERSION
    )
  except Exception as ex:
    raise HTTPException(status_code=400, detail = str(ex))

@app.get("/ready")
def ready():
  try:
    pipeline.predict(np.zeros((1, pipeline.n_features_in_)))
    return {"status" : "ready"}
  except Exception as ex:
    raise HTTPException(status_code=503, detail= "Model Not Ready")


  # Run: uvicorn app:app --host 0.0.0.0 --port 8000

Pipeline saved: Accuracy: 0.903


Optimized ML Dockerfile

Production Dockerfile with layer caching, security, and minimal image size



In [ ]:
# ============================================================
# Dockerfile - Optimized for ML Serving
# ============================================================
# Use slim base (120MB vs 900MB for full image)
FROM python:3.12-slim

# System deps for sklearn/XGBoost (libgomp = OpenMP)
RUN apt-get update && apt-get install -y \
    libgomp1 \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# LAYER CACHING: requirements first, code second
# Changing app.py won't trigger pip install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model artifacts and application code
COPY models/ ./models/
COPY app.py .

# SECURITY: non-root user
RUN useradd -m appuser && chown -R appuser /app
USER appuser

# Ensure Python output appears in docker logs
ENV PYTHONUNBUFFERED=1

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

# ============================================================
# requirements.txt (pin EXACT versions)
# ============================================================
# fastapi==0.115.0
# uvicorn==0.32.0
# scikit-learn==1.6.0
# joblib==1.4.2
# numpy==2.1.0
# pydantic==2.10.0

# ============================================================
# Build, test, deploy
# ============================================================
# Build:  docker build -t ml-api:v1 .
# Test:   docker run -p 8000:8000 ml-api:v1
# Verify: curl http://localhost:8000/health
# Deploy: docker push registry.example.com/ml-api:v1

A/B Testing with Canary Rollout

Gradual model rollout with deterministic traffic splitting and instant rollback






In [1]:
# ============================================================
# A/B TESTING: Canary deployment with rollback
# ============================================================
import os
import hashlib
import logging
import joblib
import numpy as np
from fastapi import FastAPI

app = FastAPI()
logger = logging.getLogger("ab_test")

# Load multiple versions at startup
models = {
    "v1": joblib.load("models/v1/pipeline.pkl"),
    "v2": joblib.load("models/v2/pipeline.pkl")
}

# Traffic Split via environment variable (instant rollback)
CANARY_PCT = int(os.getenv('CANARY_PCT', '10'))

def get_model_version(user_id:str) -> str:
  """Deterministic split -> same user will get same model everytime"""
  bucket = int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100
  return "v2" if bucket < CANARY_PCT else "v1"


@app.post("/predict")
def predict(request:dict):
  user_id = request.get("user_id", "anonymous")
  version = get_model_version(user_id)
  model = models[version]

  X = np.array(request.get["features"]).reshape(1, -1)
  pred = int(model.predict(X)[0])
  prob = float(model.predict_proba(X)[0,1])

  # Log for offline A/B analysis
  logger.info(f"version={version} user={user_id} pred={pred} prob={prob:.3f}")

  return {"prediction" : pred, "probability" : prob, "model_version" : version}


# ============================================================
# Rollback commands (no redeploy needed!)
# ============================================================
# Full rollback:  docker run -e CANARY_PCT=0 ml-api:latest
# 50/50 split:    docker run -e CANARY_PCT=50 ml-api:latest
# Full promotion: docker run -e CANARY_PCT=100 ml-api:latest